In [1]:
import os
import json
import pandas as pd


pd.set_option("display.max_columns", None)
# pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [2]:
base_path = "../results"

folders = {

    3: "validation_eval_service_3",
    5: "validation_eval_service_5",
    8: "validation_eval_service_8",
    10: "validation_eval_service_10"

}

In [3]:
data = {}

for k, folder_name in folders.items():

    folder_path = os.path.join(base_path, folder_name)
#     print(folder_path)

    data[k] = []

    for file_name in os.listdir(folder_path):

        if not file_name.endswith(".json"):
            continue

        if "_efficiency" in file_name:
            continue

        file_path = os.path.join(folder_path, file_name)
#         print(file_path)

        with open(file_path, "r") as f:
            rows = json.load(f)
#             print(rows)

        for row in rows:
            row = row.copy()
            row["source_file"] = file_name
            data[k].append(row)

    print(f"K={k}: loaded {len(data[k])} rows")

K=3: loaded 273 rows
K=5: loaded 339 rows
K=8: loaded 504 rows
K=10: loaded 603 rows


In [4]:
df_k3 = pd.DataFrame(data[3])
df_k5 = pd.DataFrame(data[5])
df_k8 = pd.DataFrame(data[8])
df_k10 = pd.DataFrame(data[10])

print("K=3:", df_k3.shape)
print("K=5:", df_k5.shape)
print("K=8:", df_k8.shape)
print("K=10:", df_k10.shape)

def filter_model(df, model_name):
    return df[df["model_name"] == model_name].copy()

K=3: (273, 28)
K=5: (339, 28)
K=8: (504, 28)
K=10: (603, 28)


In [5]:
MODEL = "Qwen/Qwen3-Reranker-4B"

# MODEL = "mixedbread-ai/mxbai-rerank-large-v2"
# MODEL = "BAAI/bge-reranker-v2-m3"

df_k3 = filter_model(df_k3, MODEL)
df_k5 = filter_model(df_k5, MODEL)
df_k8 = filter_model(df_k8, MODEL)
df_k10 = filter_model(df_k10, MODEL)

In [6]:
# print(len(df_k3))
# print(len(df_k5))
# print(len(df_k8))
# print(len(df_k10))

# Distribution Analysis Before threshold condition

#### to see the pattern how the model predict specialists and general


In [7]:

def distribution_summary(df):

    general = df[
        df["expected_service"] == "general"
    ]["zscore_margin"]

    specialist = df[
        df["expected_service"] != "general"
    ]["zscore_margin"]

    return pd.DataFrame(
        [
            general.describe(),
            specialist.describe()
        ],
        index=[
            "general",
            "specialist"
        ]
    )

In [8]:
distribution_summary(df_k3)

,count,mean,std,min,25%,50%,75%,max
general,50.0,1.227114,0.655589,0.0000,0.7522,1.4600,1.7590,2.1213
specialist,41.0,1.967976,0.242000,1.0228,1.9462,2.0493,2.0947,2.1213


In [9]:
distribution_summary(df_k5)

,count,mean,std,min,25%,50%,75%,max
general,50.0,0.765638,0.562908,0.0000,0.31055,0.59405,1.220025,1.9277
specialist,63.0,2.060157,0.361866,0.5804,1.94890,2.14100,2.299850,2.4694


In [10]:
distribution_summary(df_k8)

,count,mean,std,min,25%,50%,75%,max
general,50.0,0.5708,0.453956,0.0000,0.17355,0.41815,0.975125,1.5428
specialist,118.0,2.0250,0.681826,0.0388,1.85930,2.20960,2.485350,2.9111


In [11]:
distribution_summary(df_k10)

,count,mean,std,min,25%,50%,75%,max
general,50.0,0.574400,0.409387,0.0,0.24720,0.4798,0.891425,1.4982
specialist,151.0,2.086712,0.759991,0.0,1.76995,2.2617,2.635650,3.1459


In [12]:
summary = []

for k, df in {
    3: df_k3,
    5: df_k5,
    8: df_k8,
    10: df_k10,
}.items():

    general = df[
        df["expected_service"] == "general"
    ]["zscore_margin"]

    specialist = df[
        df["expected_service"] != "general"
    ]["zscore_margin"]

    summary.append({
        "K": k,
        "General Median": round(general.median(), 3),
#         "General Std": round(general.std(), 3),
        "Specialist Median": round(specialist.median(), 3),
#         "Specialist Std": round(specialist.std(), 3),
    })

summary_df = pd.DataFrame(summary)

summary_df

,K,General Median,Specialist Median
0,3,1.460,2.049
1,5,0.594,2.141
2,8,0.418,2.210
3,10,0.480,2.262


In [13]:
summary = []

for k, df in {
    3: df_k3,
    5: df_k5,
    8: df_k8,
    10: df_k10,
}.items():

    general = df[
        df["expected_service"] == "general"
    ]

    specialist = df[
        df["expected_service"] != "general"
    ]

    summary.append({
        "K": k,

        "General Margin": round(
            general["margin"].median(), 3
        ),
        "Specialist Margin": round(
            specialist["margin"].median(), 3
        ),

        "General Top Z": round(
            general["top_zscore"].median(), 3
        ),
        "Specialist Top Z": round(
            specialist["top_zscore"].median(), 3
        ),

        "General Z-Margin": round(
            general["zscore_margin"].median(), 3
        ),
        "Specialist Z-Margin": round(
            specialist["zscore_margin"].median(), 3
        ),
    })

pd.DataFrame(summary)

,K,General Margin,Specialist Margin,General Top Z,Specialist Top Z,General Z-Margin,Specialist Z-Margin
0,3,0.469,9.625,1.298,1.412,1.460,2.049
1,5,0.375,8.438,1.325,1.958,0.594,2.141
2,8,0.312,6.219,1.527,2.483,0.418,2.210
3,10,0.312,5.938,1.704,2.764,0.480,2.262


## Threshold cutting based on quartiles

In [14]:
summary = []

for k, df in {
    3: df_k3,
    5: df_k5,
    8: df_k8,
    10: df_k10,
}.items():

    general = df[
        df["expected_service"] == "general"
    ]["zscore_margin"]

    specialist = df[
        df["expected_service"] != "general"
    ]["zscore_margin"]

    summary.append({
        "K": k,

        "General Q25": round(general.quantile(0.25), 3),
        "General Q75": round(general.quantile(0.75), 3),

        "Specialist Q25": round(specialist.quantile(0.25), 3),
        "Specialist Q75": round(specialist.quantile(0.75), 3),
    })

pd.DataFrame(summary)

,K,General Q25,General Q75,Specialist Q25,Specialist Q75
0,3,0.752,1.759,1.946,2.095
1,5,0.311,1.220,1.949,2.300
2,8,0.174,0.975,1.859,2.485
3,10,0.247,0.891,1.770,2.636


In [15]:
# 0.25 - 2.5 threshold for sweeping